# m6A GRU — training-only cross-validation and held-out testing

## Goal and workflow
Train **one shared GRU** using the training partitions of dataset0, dataset1 and dataset2. Keep each input and its test predictions separate on disk.

1. Read one cleaned **train CSV and test CSV per dataset**. Do not create a new train/test split here.
2. Check biological overlap across **all** train/test files; stop if a site, transcript, known gene or supplied group crosses the boundary.
3. Run grouped five-fold CV on pooled **training rows only**. Fit scalers and class weights separately inside each training fold.
4. Compare the fixed baseline with tuning candidates using mean CV PR AUC, then ROC AUC. Freeze the winner **before** evaluating test labels.
5. Refit baseline and the best tuning configuration on all training rows only. Report their held-out test metrics, per dataset and pooled; never switch the winner based on test results.
6. Save weights, preprocessing, configurations, OOF/test predictions, metrics, plots, split checks and provenance.

**No training-set final evaluation is produced.** `grouped_OOF` is training-data validation; `held_out_test` is final testing. Tuned OOF results are selection-biased; test results are the independent estimate only if the test set has not previously influenced model choices. Old models trained on all rows cannot be reused for this test: retrain from scratch.

The feature schema and GRU architecture are unchanged. Epochs remain fixed per configuration (no early stopping). Dataset2 uses the existing binary `label`; fraction columns are never predictors. The notebook does not alter your preprocessing split or merge your six CSV files on disk.

## 1. Install once and select your kernel
Use Python 3.10 or newer. Run the install cell if these packages are missing, then restart the kernel. CPU works; PyTorch uses CUDA automatically when supported by your installation. On Ronin, select the remote Python environment in VS Code. Keep the machine running until training finishes.

In [1]:
# Uncomment once if packages are missing; restart the kernel afterwards.
%pip install "numpy>=1.24,<3" "pandas>=2,<3" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4" "torch>=2.2,<3" "joblib>=1.3,<2" ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:

from pathlib import Path
from datetime import datetime, timezone
import time
import gc, gzip, hashlib, importlib.metadata, json, random, re, shutil, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, auc, precision_score, recall_score, f1_score, accuracy_score)
from IPython.display import display
plt.rcParams.update({'figure.figsize': (8, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 20)

## 2. Set the six input files
Change `PROJECT_ROOT`, then give the actual `train_file` and `test_file` paths for each dataset. Example names below are placeholders, not required filenames; `.csv` and `.csv.gz` both work. Paths are relative to `PROJECT_ROOT` unless absolute.

Each file must contain the original cleaned feature columns: site keys, `sequence`, `n_reads_valid`, the 45 signal summaries, and binary `label`. **A split/label manifest alone is insufficient**: it must be joined to the cleaned features first. Optional `gene_id`, `group_id`, `group_source`, `dataset`/`dataset_id` and `split` columns are checked and never used as predictors. If a `split` column exists, train files may contain `train` and `validation` (validation is folded into the training pool); test files must contain only `test`.

`clean_manifest` is an optional preprocessing manifest path for provenance. Raw JSON and an all-sites prediction file are no longer required for held-out evaluation. No QC fallback scores are included in testing.

`RUN_TUNING=False` runs only the baseline. Otherwise the original six-trial search and configuration remain. Final comparison includes baseline and best tuned model, but selection is already frozen from CV. Default cost: 37 fits. For a quick pipeline check, use one epoch and disable tuning; do not treat that as the final experiment.

In [3]:
PROJECT_ROOT = Path('/home/ubuntu/project')  # EDIT
# Windows example: Path(r'D:/your/folder/Team_Project')
DATASETS = [
    dict(dataset_id='dataset0', train_file='outputs/data_cleaning/20261014T062257_386425Z/dataset0/dataset0_clean_train.csv.gz',
         test_file='outputs/data_cleaning/20261014T062257_386425Z/dataset0/dataset0_clean_test.csv.gz'),
    dict(dataset_id='dataset1', train_file='outputs/data_cleaning/20261014T062257_386425Z/dataset1/dataset1_clean_train.csv.gz',
         test_file='outputs/data_cleaning/20261014T062257_386425Z/dataset1/dataset1_clean_test.csv.gz'),
    dict(dataset_id='dataset2', train_file='outputs/data_cleaning/20261014T062257_386425Z/dataset2/dataset2_clean_train.csv.gz',
         test_file='outputs/data_cleaning/20261014T062257_386425Z/dataset2/dataset2_clean_test.csv.gz'),
]
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'gru_train_test'
SEED = 42
N_FOLDS = 5
THRESHOLD = 0.5  # Fixed before test evaluation; AUC uses continuous scores.
PREDICT_BATCH_SIZE = 1024
RUN_TUNING = True
N_TRIALS = 6
BASE_CONFIG = dict(hidden_size=32, dropout=0.2, lr=1e-3, weight_decay=1e-4,
                   batch_size=256, epochs=20, class_weighting=True)
SEARCH_SPACE = dict(hidden_size=[16, 32, 64], dropout=[0.0, 0.2, 0.4],
                    lr=[3e-4, 1e-3], weight_decay=[0.0, 1e-4, 1e-3],
                    batch_size=[128, 256], epochs=[10, 20, 30], class_weighting=[True])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def resolve_path(value):
    p = Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT / p).resolve()
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
print('Model fits planned:', N_FOLDS + 1 + (N_TRIALS*N_FOLDS + 1 if RUN_TUNING else 0))

Device: cpu | PyTorch: 2.14.1+cu130
Model fits planned: 37


## 3. Validate the supplied partitions
Only the explicitly listed features enter the model. Both train and test files must contain valid cleaned features and binary labels. Missing labels, duplicate site keys within a file, inconsistent provenance, invalid sequences and wrong split tags stop the run. Your cleaner must handle dataset2 label conversion beforehand.

Rows remain separate observations across datasets. A larger dataset contributes more training weight. Pooled metrics describe this mixture, not an equally weighted average of datasets. Within-site signal summaries can be calculated before splitting; any across-site learned preprocessing must be fitted on training only.

In [4]:
LOCATIONS = ['minus1', 'center', 'plus1']
MEASURES = ['dwell', 'signal_sd', 'signal_mean']
STATS = ['mean', 'std', 'median', 'q25', 'q75']
# Explicit ordering is essential: do not reshape the CSV's original column order.
POSITION_COLUMNS = [[f'{loc}_{measure}_{stat}' for stat in STATS for measure in MEASURES]
                    for loc in LOCATIONS]
SUMMARY_COLUMNS = [c for cols in POSITION_COLUMNS for c in cols]
KEY = ['transcript_id', 'transcript_position']

def validate_features(frame):
    required = KEY + ['sequence', 'n_reads_valid'] + SUMMARY_COLUMNS
    missing = sorted(set(required) - set(frame.columns))
    if missing: raise ValueError(f'Missing feature columns: {missing}')
    if frame.empty: raise ValueError('No sites found.')
    if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
        raise ValueError('Missing or duplicate site keys.')
    if frame.transcript_id.astype(str).str.strip().eq('').any():
        raise ValueError('Empty transcript ID.')
    pos = pd.to_numeric(frame.transcript_position, errors='raise').to_numpy(dtype=float)
    if not np.isfinite(pos).all() or (pos < 0).any() or (pos != np.floor(pos)).any():
        raise ValueError('Positions must be nonnegative integers.')
    seq = frame.sequence.astype('string')
    if not seq.str.fullmatch('[ACGT]{7}').fillna(False).all():
        raise ValueError('Expected seven-base A/C/G/T sequences.')
    if not seq.str.slice(1, 6).str.fullmatch('[AGT][AG]AC[ACT]').fillna(False).all():
        raise ValueError('Non-DRACH central motif; inspect cleaning.')
    values = frame[SUMMARY_COLUMNS + ['n_reads_valid']].to_numpy(dtype=float)
    if not np.isfinite(values).all() or (frame.n_reads_valid < 1).any():
        raise ValueError('Nonfinite summaries or invalid coverage.')

In [5]:
def read_clean_csv(path):
    if not path.is_file(): raise FileNotFoundError(f'Check input path: {path}')
    return pd.read_csv(path, dtype={'transcript_id': str, 'gene_id': str, 'group_id': str})

def load_inputs(configs):
    if not configs: raise ValueError('Add at least one dataset.')
    ids = [c['dataset_id'] for c in configs]
    if len(set(ids)) != len(ids) or any(not re.fullmatch(r'[A-Za-z0-9_-]+', x) for x in ids):
        raise ValueError('Dataset IDs must be unique and filename-safe.')
    if 'COMBINED' in ids: raise ValueError('COMBINED is reserved.')
    train_parts, tests, manifests, paths, summary = [], {}, {}, {}, []
    for c in configs:
        name = c['dataset_id']; paths[name] = {}
        for role in ['train', 'test']:
            path = resolve_path(c[role+'_file'])
            frame = read_clean_csv(path)
            validate_features(frame)
            if 'label' not in frame or not frame.label.isin([0, 1]).all():
                raise ValueError(f'{name}/{role}: need nonmissing binary label, not fractions.')
            for col in ['dataset', 'dataset_id']:
                if col in frame and not frame[col].eq(name).all():
                    raise ValueError(f'{name}/{role}: inconsistent {col}.')
            if 'split' in frame:
                allowed = ['train','validation'] if role == 'train' else ['test']
                if not frame.split.astype('string').str.strip().str.lower().isin(allowed).all():
                    raise ValueError(f'{name}/{role}: unexpected split tags; expected {allowed}.')
            frame['transcript_position'] = frame.transcript_position.astype('int64')
            frame['label'] = frame.label.astype('int64')
            frame['dataset_id'] = name
            frame['partition'] = role
            if 'gene_id' not in frame: frame['gene_id'] = pd.NA
            for col in ['gene_id', 'group_id']:
                if col in frame:
                    frame[col] = frame[col].astype('string').str.strip().replace('', pd.NA)
            paths[name][role] = str(path)
            summary.append(dict(dataset_id=name, partition=role, sites=len(frame),
                                positive_sites=int(frame.label.sum()), positive_fraction=float(frame.label.mean())))
            if role == 'train': train_parts.append(frame)
            else: tests[name] = frame.reset_index(drop=True)
        if c.get('clean_manifest'):
            manifest_path = resolve_path(c['clean_manifest'])
            manifest = json.loads(manifest_path.read_text())
            if manifest.get('max_lines') is not None:
                raise ValueError(f'{name}: manifest indicates a partial cleaning run.')
            manifests[name] = manifest; paths[name]['manifest'] = str(manifest_path)
    training = pd.concat(train_parts, ignore_index=True)
    if training.label.nunique() != 2: raise ValueError('Pooled training needs both classes.')
    return tests, training, manifests, paths, pd.DataFrame(summary)

if N_FOLDS < 2 or PREDICT_BATCH_SIZE < 1 or not 0 < THRESHOLD < 1 or N_TRIALS < 1:
    raise ValueError('Invalid run settings.')
dataset_tables, train_df, cleaning_manifests, input_paths, input_summary = load_inputs(DATASETS)
test_df = pd.concat(dataset_tables.values(), ignore_index=True)
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
input_summary.to_csv(RUN_DIR/'input_summary.csv', index=False)
display(input_summary)
print(f'Training: {len(train_df):,} rows | held-out test: {len(test_df):,} rows')
print('Outputs:', RUN_DIR)

,dataset_id,partition,sites,positive_sites,positive_fraction
0,dataset0,train,97879,4398,0.044933
1,dataset0,test,23959,1077,0.044952
2,dataset1,train,73380,5339,0.072758
3,dataset1,test,17430,1254,0.071945
4,dataset2,train,1015,870,0.857143
5,dataset2,test,308,264,0.857143


Training: 172,274 rows | held-out test: 41,697 rows
Outputs: /home/ubuntu/project/outputs/gru_train_test/20261014T063539_949196Z


## 4. Audit train/test separation, then create training-only CV folds
All datasets use shared biological identifiers. We conservatively link rows by transcript, known gene and optional `group_id` into connected components, including an unambiguous `group_id="gene:..."` when no gene column is available. Annotation keys from test data are used **only for leakage checking/group consistency**, never features, labels, scaling or fitting. Missing gene annotations limit any claim of complete gene-disjoint evaluation.

If a biological component spans train and test, the notebook writes overlap diagnostics and stops. Repair the upstream shared split; do not silently remove test rows or independently resplit each dataset. CV groups are then taken only for training rows. Baseline and all candidates share identical frozen folds.

In [6]:
# Union-find uses identifiers only; labels are never involved.
all_keys = pd.concat([train_df, test_df], ignore_index=True)
parent = {}
def root(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def link(a, b):
    ra, rb = root(a), root(b)
    if ra != rb: parent[rb] = ra
known_pairs = []
for row in all_keys.itertuples(index=False):
    tx = 'transcript:' + row.transcript_id
    root(tx)
    gene = row.gene_id
    supplied = getattr(row, 'group_id', pd.NA)
    if pd.notna(supplied) and str(supplied).startswith('gene:'):
        group_gene = str(supplied)[5:]
        if pd.notna(gene) and str(gene) != group_gene:
            raise ValueError('gene_id conflicts with gene-based group_id.')
        gene = group_gene
    if pd.notna(gene):
        known_pairs.append((row.transcript_id, str(gene)))
        link(tx, 'gene:' + str(gene))
    if pd.notna(supplied): link(tx, 'supplied:' + str(supplied))
known = pd.DataFrame(known_pairs, columns=['transcript_id','gene_id']).drop_duplicates()
if known.groupby('transcript_id').gene_id.nunique().gt(1).any():
    raise ValueError('Conflicting transcript-to-gene annotations.')
components = all_keys.transcript_id.map(lambda t: root('transcript:' + t))
groups = components.iloc[:len(train_df)].reset_index(drop=True)
test_groups = components.iloc[len(train_df):].reset_index(drop=True)
shared_groups = set(groups) & set(test_groups)
shared_sites = train_df[KEY].drop_duplicates().merge(test_df[KEY].drop_duplicates(), on=KEY)
shared_transcripts = set(train_df.transcript_id) & set(test_df.transcript_id)
audit = dict(shared_sites=len(shared_sites), shared_transcripts=len(shared_transcripts),
             shared_biological_groups=len(shared_groups), train_rows=len(train_df), test_rows=len(test_df))
(RUN_DIR/'split_audit.json').write_text(json.dumps(audit, indent=2))
all_keys[['dataset_id','partition']+KEY+['gene_id']].assign(cv_group=components).to_csv(
    RUN_DIR/'input_split_assignments.csv.gz', index=False)
if shared_groups or len(shared_sites) or shared_transcripts:
    all_keys.loc[components.isin(shared_groups), ['dataset_id','partition']+KEY+['gene_id']].to_csv(
        RUN_DIR/'train_test_overlap.csv', index=False)
    raise ValueError(f'Train/test biological overlap: {audit}. Fix upstream shared splits; see train_test_overlap.csv.')
print('Train/test separation passed:', audit)
if groups.nunique() < N_FOLDS: raise ValueError('Too few independent groups for configured folds.')
grouping_summary = dict(known_gene_rows=int(train_df.gene_id.notna().sum()),
                        missing_gene_rows=int(train_df.gene_id.isna().sum()), groups=int(groups.nunique()))
print('Grouping:', grouping_summary)
overlaps = train_df.loc[train_df.duplicated(KEY, keep=False), ['dataset_id']+KEY+['label']]
overlaps.to_csv(RUN_DIR/'cross_dataset_shared_sites.csv', index=False)
conflicts = overlaps.groupby(KEY).label.nunique().gt(1).sum() if len(overlaps) else 0
print(f'Shared-site rows: {len(overlaps):,}; shared sites with disagreeing labels: {conflicts:,}')
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(cv.split(np.zeros(len(train_df)), train_df.label, groups=groups))
fold_id = np.full(len(train_df), -1, dtype=int)
fold_summary = []
for number, (tr, va) in enumerate(folds, 1):
    if set(groups.iloc[tr]) & set(groups.iloc[va]): raise ValueError('Group leakage.')
    if set(train_df.iloc[tr].transcript_id) & set(train_df.iloc[va].transcript_id):
        raise ValueError('Transcript leakage.')
    for role, idx in [('train',tr),('validation',va)]:
        if train_df.iloc[idx].label.nunique() != 2:
            raise ValueError(f'Fold {number} {role} lacks a class. Review groups or reduce N_FOLDS.')
        fold_summary.append(dict(fold=number, role=role, sites=len(idx),
                                 positive_fraction=float(train_df.iloc[idx].label.mean())))
    if (fold_id[va] != -1).any(): raise ValueError('Repeated validation assignment.')
    fold_id[va] = number
assert (fold_id > 0).all()
assignments = train_df[['dataset_id']+KEY+['gene_id','label']].copy()
assignments['cv_group'] = groups.to_numpy(); assignments['validation_fold'] = fold_id
assignments.to_csv(RUN_DIR/'cv_fold_assignments.csv.gz', index=False)
display(pd.DataFrame(fold_summary))
pd.DataFrame(fold_summary).to_csv(RUN_DIR/'cv_fold_summary.csv', index=False)

ValueError: Train/test biological overlap: {'shared_sites': 0, 'shared_transcripts': 7, 'shared_biological_groups': 1, 'train_rows': 172274, 'test_rows': 41697}. Fix upstream shared splits; see train_test_overlap.csv.

## 5. Preprocessing fitted only on training rows
At each of three ordered positions (−1, centre, +1), the RNN receives 15 summaries: three measurements × five statistics. The input tensor is `(sites, 3, 15)`. A fixed A/C/G/T encoding contributes 28 additional features; log-read-coverage contributes one more. IDs and labels never enter these arrays.

Within-site read summaries are already calculated by the cleaner and can be reused. Across-site scaling is fitted afresh on each training fold, then applied to validation rows. The final models have separate scalers fitted on all training-partition rows, never test rows.

In [ ]:
def raw_arrays(frame):
    sequence = np.stack([frame[cols].to_numpy(dtype=np.float64)
                         for cols in POSITION_COLUMNS], axis=1)
    base_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    indices = np.array([[base_index[b] for b in seq] for seq in frame.sequence])
    bases = np.eye(4, dtype=np.float32)[indices].reshape(len(frame), 28)
    coverage = np.log1p(frame[['n_reads_valid']].to_numpy(dtype=np.float64))
    return sequence, bases, coverage


def fit_preprocessing(frame):
    seq, _, coverage = raw_arrays(frame)
    return {'sequence_scaler': StandardScaler().fit(seq.reshape(len(frame), 45)),
            'coverage_scaler': StandardScaler().fit(coverage)}

def transform(frame, preprocessing):
    seq, bases, cov = raw_arrays(frame)
    seq = preprocessing['sequence_scaler'].transform(seq.reshape(len(frame), 45)).reshape(-1, 3, 15)
    aux = np.concatenate([bases, preprocessing['coverage_scaler'].transform(cov)], axis=1)
    seq, aux = seq.astype('float32'), aux.astype('float32')
    if not np.isfinite(seq).all() or not np.isfinite(aux).all():
        raise ValueError('Nonfinite transformed values; inspect extreme inputs.')
    return seq, aux

## 6. Model architecture and imbalance handling
One GRU layer, followed by a small dense head, produces one logit per transcript **site**. Sigmoid converts it to a score. This is a summary-based GRU, not an LSTM or read-level multiple-instance model.

Use AdamW, gradient clipping at 1.0 and weighted binary cross-entropy. `pos_weight = negatives / positives` uses only the current training population. No validation resampling occurs. Epoch count is fixed for each configuration; the scored validation fold does not choose a checkpoint. Tuning trains new configurations from scratch rather than continuing the baseline weights.

In [ ]:
class SiteGRU(nn.Module):
    def __init__(self, hidden_size=32, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=15, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden_size + 29, 32), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, seq, aux):
        _, hidden = self.gru(seq)
        return self.head(torch.cat([hidden[-1], aux], dim=1)).squeeze(1)

def make_loader(seq, aux, batch_size, labels=None, shuffle=False, seed=SEED):
    tensors = [torch.from_numpy(seq), torch.from_numpy(aux)]
    if labels is not None: tensors.append(torch.from_numpy(labels))
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, generator=torch.Generator().manual_seed(seed))

@torch.no_grad()
def predict_scores(model, seq, aux, batch_size=512):
    model.eval()
    result = [torch.sigmoid(model(s.to(DEVICE), a.to(DEVICE))).cpu().numpy()
              for s, a in make_loader(seq, aux, batch_size)]
    return np.concatenate(result)

def fit_model(seq, aux, labels, config, seed=SEED):
    seed_everything(seed)
    model = SiteGRU(config['hidden_size'], config['dropout']).to(DEVICE)
    positive_weight = float((labels == 0).sum() / (labels == 1).sum()) if config['class_weighting'] else 1.0
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    batches = make_loader(seq, aux, config['batch_size'], labels, shuffle=True, seed=seed)
    history = []
    for epoch in range(1, config['epochs'] + 1):
        model.train(); total_loss = 0.0
        for s, a, target in batches:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(s.to(DEVICE), a.to(DEVICE)), target.to(DEVICE))
            if not torch.isfinite(loss): raise RuntimeError('Nonfinite loss.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(target)
        history.append({'epoch': epoch, 'weighted_train_loss': total_loss / len(labels)})
        if epoch == 1 or epoch % 5 == 0 or epoch == config['epochs']:
            print(f'Epoch {epoch:02d}/{config["epochs"]}: loss={history[-1]["weighted_train_loss"]:.4f}', flush=True)
    return model, pd.DataFrame(history)

## 7. Metrics and training-only CV
ROC AUC and trapezoidal PR AUC use continuous scores. Average precision (AP) is recorded separately because its integration convention differs. Precision, recall, F1 and accuracy use the fixed threshold 0.5. The test set never selects the threshold. Both AUCs are unavailable for a one-class dataset.

Each results table has one row per dataset and a **COMBINED** row calculated from pooled rows, not the average of dataset scores. CV produces OOF predictions for training rows only, with individual fold metrics and loss histories. These predictions can support ensemble development using training data. Test predictions are reserved for the frozen ensemble's final evaluation.

In [ ]:
def metric_row(y, scores):
    y = np.asarray(y, dtype=int); scores = np.asarray(scores, dtype=float)
    if len(y) != len(scores) or not np.isfinite(scores).all() or ((scores<0)|(scores>1)).any():
        raise ValueError('Invalid scores or score/label length mismatch.')
    if not np.isin(y, [0,1]).all(): raise ValueError('Metrics require binary labels.')
    result = dict(n_scored_sites=len(y), positive_fraction=float(y.mean()) if len(y) else np.nan,
                  ROC_AUC=np.nan, PR_AUC_trapezoidal=np.nan, average_precision=np.nan,
                  precision=np.nan, recall=np.nan, F1=np.nan, accuracy=np.nan,
                  status='no binary labels')
    if not len(y): return result
    pred = (scores >= THRESHOLD).astype(int)
    result.update(precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0), F1=f1_score(y,pred,zero_division=0),
        accuracy=accuracy_score(y,pred))
    if len(np.unique(y)) < 2:
        result['status'] = 'one class; ROC/PR AUC unavailable'
        return result
    precision, recall, _ = precision_recall_curve(y, scores)
    result.update(ROC_AUC=roc_auc_score(y,scores), PR_AUC_trapezoidal=auc(recall,precision),
                  average_precision=average_precision_score(y,scores), status='ok')
    return result

def evaluate_tables(frames, scores_by_dataset, model_name, evaluation):
    records, pooled_y, pooled_scores = [], [], []
    for name, frame in frames.items():
        scores = np.asarray(scores_by_dataset[name])
        if len(scores) != len(frame): raise ValueError(f'{name}: score length mismatch.')
        mask = frame.label.notna().to_numpy()
        y = frame.loc[mask,'label'].to_numpy(dtype=int)
        s = scores[mask]
        records.append(dict(model=model_name, evaluation=evaluation, dataset_id=name,
                            n_prediction_sites=len(frame), **metric_row(y,s)))
        pooled_y.append(y); pooled_scores.append(s)
    records.append(dict(model=model_name, evaluation=evaluation, dataset_id='COMBINED',
                        n_prediction_sites=sum(map(len,frames.values())),
                        **metric_row(np.concatenate(pooled_y), np.concatenate(pooled_scores))))
    return pd.DataFrame(records)

def run_cv(config, experiment_name):
    folder = RUN_DIR / experiment_name; folder.mkdir(exist_ok=False)
    oof = np.full(len(train_df), np.nan); records = []
    for number,(tr,va) in enumerate(folds,1):
        print(f'\n{experiment_name}: fold {number}/{N_FOLDS}', flush=True)
        prep = fit_preprocessing(train_df.iloc[tr])
        train_seq, train_aux = transform(train_df.iloc[tr],prep)
        val_seq, val_aux = transform(train_df.iloc[va],prep)
        fit_started = time.perf_counter()
        model,history = fit_model(train_seq,train_aux,
            train_df.iloc[tr].label.to_numpy(dtype=np.float32),config,seed=SEED+number)
        scores = predict_scores(model,val_seq,val_aux,PREDICT_BATCH_SIZE)
        oof[va] = scores
        records.append(dict(fold=number, fit_and_predict_seconds=time.perf_counter()-fit_started, **metric_row(train_df.iloc[va].label,scores)))
        history.to_csv(folder/f'fold_{number}_history.csv',index=False)
        del model,prep,train_seq,train_aux,val_seq,val_aux
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    if not np.isfinite(oof).all(): raise ValueError('Incomplete OOF scores.')
    fold_metrics = pd.DataFrame(records)
    fold_metrics.to_csv(folder/'fold_metrics.csv',index=False)
    output = assignments.copy(); output['score'] = oof
    output.to_csv(folder/'oof_predictions.csv.gz',index=False)
    # Include prediction-only datasets as explicit unavailable rows, not fabricated CV predictions.
    frames, by_dataset = {}, {}
    for c in DATASETS:
        name=c['dataset_id']; mask=train_df.dataset_id.eq(name).to_numpy()
        frames[name]=train_df.loc[mask].reset_index(drop=True)
        by_dataset[name]=oof[mask]
    table=evaluate_tables(frames,by_dataset,experiment_name,'grouped_OOF')
    table.to_csv(folder/'dataset_metrics.csv',index=False)
    objective=dict(PR_AUC_trapezoidal=float(fold_metrics.PR_AUC_trapezoidal.mean()),
                   ROC_AUC=float(fold_metrics.ROC_AUC.mean()))
    return dict(config=config.copy(),oof=oof,fold_metrics=fold_metrics,metrics=table,objective=objective)

## 8. Define final train-only refitting and test-only export
These functions are called **after CV selection is frozen**. Each final model's scaler, class weight and weights use the training partition only. Saved prediction CSVs contain only the held-out test sites, with the handout's three columns. Separate detailed CSVs retain labels, model, threshold, dataset and site IDs for audit/ensemble alignment.

These are **test-subset predictions**, not complete original-JSON submission files. For leaderboard or Task 2 data, use the saved model with the same preprocessing on those separate inputs; full raw-JSON coverage still needs your prediction script. No missing/QC-failed site gets a fabricated test score.

In [ ]:
def fit_train_and_evaluate_test(config, name):
    folder = RUN_DIR/name; folder.mkdir(exist_ok=True)
    started = time.perf_counter()
    prep = fit_preprocessing(train_df)
    seq, aux = transform(train_df, prep)
    model, history = fit_model(seq, aux, train_df.label.to_numpy(dtype=np.float32), config, seed=SEED)
    fit_seconds = time.perf_counter()-started
    history.to_csv(folder/'train_refit_history.csv', index=False)
    del seq, aux
    checkpoint = dict(state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
                      config=config, position_columns=POSITION_COLUMNS,
                      threshold=THRESHOLD, architecture='SiteGRU',
                      training_prevalence=float(train_df.label.mean()), training_partition='train_only')
    torch.save(checkpoint, folder/'model.pt')
    joblib.dump(prep, folder/'preprocessing.joblib')
    (folder/'config.json').write_text(json.dumps(config, indent=2))
    pred_dir = folder/'test_predictions'; pred_dir.mkdir(exist_ok=True)
    scores_by_dataset, exports = {}, []
    started = time.perf_counter()
    for dataset, frame in dataset_tables.items():
        seq, aux = transform(frame, prep)
        scores = predict_scores(model, seq, aux, PREDICT_BATCH_SIZE)
        scores_by_dataset[dataset] = scores
        path = pred_dir/f'{dataset}_test_predictions.csv'
        frame[KEY].assign(score=scores.astype(float)).to_csv(path,index=False,float_format='%.10g')
        cols = ['dataset_id']+KEY+['gene_id','label']
        frame[cols].assign(score=scores, predicted_label=(scores>=THRESHOLD).astype(int),
                           model=name, partition='test', threshold=THRESHOLD).to_csv(
            pred_dir/f'{dataset}_test_predictions_detailed.csv', index=False)
        exports.append(dict(dataset_id=dataset, file=str(path), rows=len(frame), partition='test'))
        del seq, aux
    predict_seconds = time.perf_counter()-started
    metrics = evaluate_tables(dataset_tables, scores_by_dataset, name, 'held_out_test')
    metrics.to_csv(folder/'test_dataset_metrics.csv', index=False)
    timing = dict(model=name, refit_seconds=fit_seconds, test_transform_predict_export_seconds=predict_seconds,
                  training_rows=len(train_df), test_rows=len(test_df))
    (folder/'timing.json').write_text(json.dumps(timing,indent=2))
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dict(metrics=metrics, scores=scores_by_dataset, exports=exports, timing=timing)

## 9. Baseline CV on training rows only
Train the original fixed configuration on the shared grouped folds. No final refitting or test predictions occur yet.

In [ ]:
baseline_cv=run_cv(BASE_CONFIG,'baseline')
print('Baseline training-only grouped OOF evaluation:')
display(baseline_cv['metrics'].round(4))

## 10. Hyperparameter tuning on training-only CV
This random search varies hidden size, dropout, learning rate, weight decay, batch size and fixed training epochs. Weighted loss remains enabled. No probability threshold is tuned because AUC metrics use scores across thresholds.

Rank candidates by mean validation-fold trapezoidal PR AUC, then mean ROC AUC. The same folds, fold seeds, rows and preprocessing rules make comparisons consistent. Six trials is an initial compute budget, not an exhaustive search. Inspect the per-dataset results for tradeoffs; a pooled winner may worsen a smaller dataset.

With `RUN_TUNING=False`, this section is skipped and the baseline is selected. With the default settings, 37 models (including the two train-only refits after selection) are fitted sequentially. There is no nested-CV independence claim for the selected tuned configuration.

In [ ]:
def selection_key(result):
    obj=result['objective']
    return (obj['PR_AUC_trapezoidal'],obj['ROC_AUC'])

tuned_cv=tuned_fit=None
trial_rows=[]
if RUN_TUNING:
    candidates=list(ParameterSampler(SEARCH_SPACE,n_iter=N_TRIALS,random_state=SEED))
    for number,config in enumerate(candidates,1):
        config={k:(v.item() if isinstance(v,np.generic) else v) for k,v in config.items()}
        print(f'\nTuning candidate {number}/{len(candidates)}: {config}',flush=True)
        result=run_cv(config,f'trial_{number:02d}')
        trial_rows.append(dict(trial=number,**result['objective'],**config))
        if tuned_cv is None or selection_key(result)>selection_key(tuned_cv): tuned_cv=result
    tuning_table=pd.DataFrame(trial_rows).sort_values(['PR_AUC_trapezoidal','ROC_AUC'],ascending=False)
    tuning_table.to_csv(RUN_DIR/'tuning_leaderboard.csv',index=False)
    display(tuning_table.round(4))
    # Keep tuned artifacts named consistently; OOF scores were produced by the selected trial.
    tuned_cv['metrics']=tuned_cv['metrics'].assign(model='tuned')
    tuned_folder=RUN_DIR/'tuned'; tuned_folder.mkdir()
    tuned_cv['fold_metrics'].to_csv(tuned_folder/'fold_metrics.csv',index=False)
    tuned_cv['metrics'].to_csv(tuned_folder/'dataset_metrics.csv',index=False)
    tuned_oof=assignments.copy(); tuned_oof['score']=tuned_cv['oof']
    tuned_oof.to_csv(tuned_folder/'oof_predictions.csv.gz',index=False)
    print('Tuned grouped OOF — used for selection, potentially optimistic:')
    display(tuned_cv['metrics'].round(4))
else:
    print('Tuning disabled; baseline remains available for prediction.')

## 11. Freeze the selection, then evaluate the test partitions
The winner is written to `model_selection.csv` and `selection.json` before any test prediction. The fixed rule is mean fold PR AUC, then ROC AUC, with exact ties retaining baseline. Both baseline and best tuned configuration are then refitted using all training rows and compared on test rows for reporting. **Test results never change the selected model.** Do not repeatedly tune after inspecting these test results.

`delta_*` means tuned minus baseline. Positive deltas are observed differences, not a significance test. Training-only OOF and held-out testing are clearly separated in all saved tables.

In [ ]:
selected_name = 'tuned' if tuned_cv is not None and selection_key(tuned_cv)>selection_key(baseline_cv) else 'baseline'
selection_rows=[dict(model='baseline',**baseline_cv['objective'])]
if tuned_cv is not None: selection_rows.append(dict(model='tuned',**tuned_cv['objective']))
selection_table=pd.DataFrame(selection_rows)
selection_table['selected']=selection_table.model.eq(selected_name)
selection_table.to_csv(RUN_DIR/'model_selection.csv',index=False)
display(selection_table.round(4))
print('Selected:',selected_name,'using mean fold PR AUC, then ROC AUC.')
if tuned_cv is not None:
    delta_pr=tuned_cv['objective']['PR_AUC_trapezoidal']-baseline_cv['objective']['PR_AUC_trapezoidal']
    delta_roc=tuned_cv['objective']['ROC_AUC']-baseline_cv['objective']['ROC_AUC']
    print(f'Mean fold changes: PR AUC {delta_pr:+.4f}; ROC AUC {delta_roc:+.4f}.')
    if delta_pr*delta_roc<0: print('Metric tradeoff: one AUC improves while the other decreases.')
(RUN_DIR/'selection.json').write_text(json.dumps(dict(selected_model=selected_name, source='training_CV_only', frozen_before_test=True),indent=2))
baseline_fit=fit_train_and_evaluate_test(BASE_CONFIG,'baseline')
tuned_fit=fit_train_and_evaluate_test(tuned_cv['config'],'tuned') if tuned_cv is not None else None
metric_tables=[baseline_cv['metrics'],baseline_fit['metrics']]
if tuned_cv is not None: metric_tables += [tuned_cv['metrics'],tuned_fit['metrics']]
all_metrics=pd.concat(metric_tables,ignore_index=True)
all_metrics.to_csv(RUN_DIR/'all_model_dataset_metrics.csv',index=False)
display(all_metrics.round(4))

if tuned_cv is not None:
    baseline_rows=all_metrics.loc[all_metrics.model.eq('baseline')]
    tuned_rows=all_metrics.loc[all_metrics.model.eq('tuned')]
    comparison=baseline_rows.merge(tuned_rows,on=['evaluation','dataset_id'],suffixes=('_baseline','_tuned'),validate='one_to_one')
    for metric in ['ROC_AUC','PR_AUC_trapezoidal','average_precision']:
        comparison['delta_'+metric]=comparison[metric+'_tuned']-comparison[metric+'_baseline']
    columns=['evaluation','dataset_id','n_scored_sites_baseline']
    for metric in ['ROC_AUC','PR_AUC_trapezoidal','average_precision']:
        columns += [metric+'_baseline',metric+'_tuned','delta_'+metric]
    comparison=comparison[columns]
    comparison.to_csv(RUN_DIR/'baseline_vs_tuned.csv',index=False)
    display(comparison.round(4))

else:
    comparison=None

## 12. Visualise validation and test results separately
ROC/PR curves show baseline and tuned scores on identical rows for each dataset and pooled data. The PR reference line is the evaluated population's positive prevalence. Tuned OOF curves were involved in selection; held-out test curves describe final performance.

Additional figures show the selected model's test confusion matrices, test metrics by dataset, CV fold variability and final refit training losses. The training-loss plots do not show test loss or choose epochs. Standard deviation across CV folds is descriptive variability, not a confidence interval.

In [ ]:
def plot_model_comparison(evaluation):
    if evaluation=='grouped_OOF':
        frame=train_df
        predictions={'baseline':baseline_cv['oof']}
        if tuned_cv is not None: predictions['tuned']=tuned_cv['oof']
    else:
        frame=pd.concat(dataset_tables.values(),ignore_index=True)
        predictions={'baseline':np.concatenate([baseline_fit['scores'][n] for n in dataset_tables])}
        if tuned_fit is not None: predictions['tuned']=np.concatenate([tuned_fit['scores'][n] for n in dataset_tables])
    for name in [c['dataset_id'] for c in DATASETS]+['COMBINED']:
        mask=frame.label.notna().to_numpy()
        if name!='COMBINED': mask &= frame.dataset_id.eq(name).to_numpy()
        y=frame.loc[mask,'label'].to_numpy(dtype=int)
        if len(np.unique(y))<2: continue
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for model_name,scores in predictions.items():
            s=scores[mask]; m=metric_row(y,s)
            colour,style=('#356995','-') if model_name=='baseline' else ('#b86b22','--')
            fpr,tpr,_=roc_curve(y,s); precision,recall,_=precision_recall_curve(y,s)
            axes[0].plot(fpr,tpr,color=colour,linestyle=style,label=f'{model_name}: AUC={m["ROC_AUC"]:.3f}')
            axes[1].plot(recall,precision,color=colour,linestyle=style,label=f'{model_name}: PR AUC={m["PR_AUC_trapezoidal"]:.3f}, AP={m["average_precision"]:.3f}')
        axes[0].plot([0,1],[0,1],':',color='gray',label='Random ranking')
        axes[1].axhline(y.mean(),linestyle=':',color='gray',label=f'Prevalence={y.mean():.3f}')
        axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='ROC')
        axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision–recall')
        for ax in axes: ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.legend(fontsize=9)
        fig.suptitle(f'{name} | {evaluation} | {len(y):,} binary-labelled sites')
        fig.tight_layout(); fig.savefig(RUN_DIR/f'{evaluation}_{name}_curves.png',dpi=150,bbox_inches='tight')
        plt.show(); plt.close(fig)

plot_model_comparison('grouped_OOF')
plot_model_comparison('held_out_test')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
chosen_fit = baseline_fit if selected_name == 'baseline' else tuned_fit
for name, frame in dataset_tables.items():
    scores = chosen_fit['scores'][name]
    fig, ax = plt.subplots(figsize=(5.5,4.5))
    ConfusionMatrixDisplay(confusion_matrix(frame.label, scores>=THRESHOLD, labels=[0,1]),
                           display_labels=['Unmodified','Modified']).plot(ax=ax,colorbar=False,cmap='Blues')
    ax.set_title(f'{name}: {selected_name} held-out test\nThreshold = {THRESHOLD}')
    fig.tight_layout(); fig.savefig(RUN_DIR/f'test_{name}_confusion_matrix.png',dpi=150,bbox_inches='tight')
    plt.show(); plt.close(fig)
fig, axes = plt.subplots(1,2,figsize=(12,4.5))
chosen_fit['metrics'].set_index('dataset_id')[['ROC_AUC','PR_AUC_trapezoidal','average_precision']].plot.bar(ax=axes[0])
axes[0].set(title=f'{selected_name}: held-out test',xlabel='Dataset',ylabel='Score',ylim=(0,1))
for result, name in [(baseline_cv,'baseline'),(tuned_cv,'tuned')]:
    if result is not None:
        axes[1].plot(result['fold_metrics']['fold'],result['fold_metrics'].PR_AUC_trapezoidal,
                     'o-' if name=='baseline' else 's--',label=name)
axes[1].set(title='Training-only CV fold variability',xlabel='Validation fold',ylabel='Trapezoidal PR AUC',ylim=(0,1))
axes[1].set_xticks(range(1,N_FOLDS+1))
axes[1].legend()
fig.tight_layout(); fig.savefig(RUN_DIR/'test_metrics_and_cv_variability.png',dpi=150,bbox_inches='tight')
plt.show(); plt.close(fig)
fig, ax = plt.subplots(figsize=(8,4))
for name in ['baseline'] + (['tuned'] if tuned_fit is not None else []):
    history = pd.read_csv(RUN_DIR/name/'train_refit_history.csv')
    ax.plot(history.epoch,history.weighted_train_loss,label=name)
ax.set(title='Final refits: training partition only',xlabel='Epoch',ylabel='Weighted training BCE loss')
ax.legend(); fig.tight_layout(); fig.savefig(RUN_DIR/'train_refit_losses.png',dpi=150,bbox_inches='tight')
plt.show(); plt.close(fig)
cv_summary = pd.concat([r['fold_metrics'].assign(model=name) for r,name in
                       [(baseline_cv,'baseline'),(tuned_cv,'tuned')] if r is not None])
cv_summary.groupby('model')[['ROC_AUC','PR_AUC_trapezoidal','average_precision']].agg(['mean','std']).to_csv(
    RUN_DIR/'cv_metric_mean_std.csv')


## 13. Save the frozen winner and experiment provenance
`selected/` contains the trained weights and preprocessing together, its final held-out test metrics, training OOF predictions and separate test predictions for each dataset. Keep dataset/site keys when aligning models for ensemble work. All output files live in a timestamped run directory.

The run manifest records six input file hashes, configs, packages, seed, device, split checks, counts, timings and limitations. No files are merged or rewritten in the preprocessing directory.

In [ ]:
selected_dir=RUN_DIR/'selected'; selected_dir.mkdir(exist_ok=False)
for filename in ['model.pt','preprocessing.joblib','config.json']:
    shutil.copy2(RUN_DIR/selected_name/filename,selected_dir/filename)
shutil.copytree(RUN_DIR/selected_name/'test_predictions',selected_dir/'test_predictions')
chosen_fit=baseline_fit if selected_name=='baseline' else tuned_fit
export_summary=pd.DataFrame(chosen_fit['exports'])
export_summary['file']=[str(selected_dir/'test_predictions'/Path(p).name) for p in export_summary.file]
export_summary.to_csv(selected_dir/'export_summary.csv',index=False)
display(export_summary)
chosen_fit['metrics'].to_csv(selected_dir/'test_dataset_metrics.csv',index=False)
shutil.copy2(RUN_DIR/selected_name/'oof_predictions.csv.gz', selected_dir/'train_oof_predictions.csv.gz')
pd.DataFrame([f['timing'] for f in [baseline_fit,tuned_fit] if f is not None]).to_csv(RUN_DIR/'final_fit_timings.csv',index=False)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
file_provenance={}
for name,paths in input_paths.items():
    file_provenance[name]={role:dict(path=path,sha256=sha256_file(Path(path)))
                          for role,path in paths.items() if path is not None}
manifest=dict(created_utc=datetime.now(timezone.utc).isoformat(),selected_model=selected_name,
    selection_rule='mean fold PR_AUC_trapezoidal, then mean fold ROC_AUC; exact tie retains baseline',
    seed=SEED,n_folds=N_FOLDS,threshold=THRESHOLD,baseline_config=BASE_CONFIG,
    tuned_config=tuned_cv['config'] if tuned_cv else None,search_space=SEARCH_SPACE,
    run_tuning=RUN_TUNING,n_trials=N_TRIALS,grouping=grouping_summary,
    test_rows=len(test_df),split_audit=audit,training_rows=len(train_df),training_positive_fraction=float(train_df.label.mean()),
    input_paths=file_provenance,cleaning_manifests=cleaning_manifests,
    packages={p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','torch','matplotlib','joblib']},
    python=sys.version,device=str(DEVICE),
    limitations=['Held-out estimates require test data not previously used for model decisions.',
                 'Tuned CV scores reuse folds involved in selection; independent results are held_out_test.',
                 'Evaluation of fraction-derived binary labels reflects the conversion rule recorded by the cleaner, not fraction prediction.',
                 'Unknown gene annotations use transcript grouping; no complete gene-disjoint claim.',
                 'Test exports cover clean held-out labelled sites only, not the complete raw input.'])
(RUN_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print('Selected prediction CSVs:',selected_dir/'test_predictions')
print('All experiment tables:',RUN_DIR)

## 14. Reload the model later and predict another cleaned dataset
Run Sections 1, 2, 3's feature-definition cell, 5 and 6 to define imports, tensor ordering, preprocessing and the model before using this example. Set `saved_dir` to the selected model directory from a completed run. Loading `joblib` files is appropriate only for artifacts you created or otherwise trust.

For a new raw JSON, first run the same cleaning notebook, then provide its full cleaning output. This notebook operates on cleaned summaries. The final project separately requires executable raw-JSON training/prediction scripts and benchmarking against m6Anet; those are not replaced by this notebook.

In [ ]:
# Example: uncomment after editing the paths.
# saved_dir = Path('/path/to/completed/run/selected')
# checkpoint = torch.load(saved_dir/'model.pt', map_location='cpu', weights_only=True)
# reloaded = SiteGRU(checkpoint['config']['hidden_size'], checkpoint['config']['dropout']).to(DEVICE)
# reloaded.load_state_dict(checkpoint['state_dict'])
# preprocessing = joblib.load(saved_dir/'preprocessing.joblib')
# new_sites = pd.read_csv('/path/to/new/clean_signal_sites.csv.gz', dtype={'transcript_id': str})
# validate_features(new_sites)
# seq, aux = transform(new_sites, preprocessing)
# scores = predict_scores(reloaded, seq, aux, PREDICT_BATCH_SIZE)
# predictions = new_sites[KEY].copy(); predictions['score'] = scores.astype(float)
# predictions.to_csv('/path/to/new_clean_site_predictions.csv', index=False)
# For raw-JSON submission, verify original key spelling and full coverage separately.

## 15. How to run and interpret the output
1. Set the six paths in Section 2. Files must contain signal features, not just the split manifest. The original feature schema is required; this authoring session did not include your new cleaned CSVs.
2. Restart the kernel and **Run All**. This retrains models from scratch; old all-data checkpoints are unsuitable for independent testing. For prediction-only use later, run the definition cells listed in Section 14 and load the saved checkpoint; skip the CV/training cells.
3. If overlap checking stops, inspect `train_test_overlap.csv` and fix your upstream split consistently across datasets. Do not change splits based on model scores.
4. Use `selected/test_dataset_metrics.csv` for the final GRU results. `all_model_dataset_metrics.csv` includes both validation and test rows, explicitly labelled. `model_selection.csv` records the CV-only choice.
5. Curves and diagnostic figures are saved as PNGs in the run directory and shown inline. Each experiment records its fold histories, fold metrics and OOF predictions; final refits record training histories, timings, weights and scalers.
6. Test-only CSVs in `selected/test_predictions/` are appropriate for held-out comparison. They are **not** full raw-JSON leaderboard submissions. Task 2 and the handout's standalone prediction script still require separate unseen input processing.
7. Preserve the selected checkpoint with its preprocessing file and feature order. Fraction-derived labels reflect your conversion rule; weighted sigmoid scores are not guaranteed calibrated probabilities.

### Sources and authoring validation
Modified from the supplied `m6a_gru_multiple_datasets_training.ipynb`, with requirements checked against `handout_project2_RNAModifications.html`. The original architecture, feature order, CV objective and default search space were retained.

Real split datasets were not supplied. Authoring validation ran all executable cells sequentially on synthetic cleaned features (with a plain-text display substitute because this environment lacks IPython/nbclient), checked both tuning-enabled and baseline-only paths, inspected saved plots, rejected cross-split overlap and wrong split tags, and verified checkpoint predictions; synthetic scores do not estimate your model's performance. Run the notebook with your six actual files to obtain real results. Record AI assistance and verified corrections for the assignment.